# Phase 2R - Does the gap come from duplication, or from grouping?

**Run on:** Kaggle or Colab, free T4. ~45-60 min for 30 training runs.

---

### The question

The similarity-grouped protocol scores 24-37 balanced-accuracy points below the
image-level protocol. That is consistent with two different stories.

**Leakage.** Grouping pulls near-duplicate images off the train/test boundary,
so the image-level number was inflated by near-copies of test images sitting in
training. This is the paper's claim.

**A grouping artefact.** Assigning whole blocks of images to one side of the
split makes test folds systematically unlike training folds for reasons
unrelated to duplication: correlated fold composition, fewer effective
independent samples, higher variance. Any blocked partition would do this,
whatever the blocks contained.

Both predict a drop. They are distinguished by a third arm.

### The control

Construct groups that are **random but size-matched**: take the exact multiset
of real similarity-group sizes and assign images to them at random. Then
partition with the **same splitter** used for the real groups.

| | blocked structure | near-duplicates separated |
|---|---|---|
| image-level | no | no |
| **random-grouped** | **yes** | **no** |
| similarity-grouped | yes | yes |

Random-grouped shares the blocking with the real protocol and shares the
duplicate-scattering with image-level, so it isolates one factor from the other.

### What each outcome means, fixed in advance

- **random-grouped ~ image-level, both >> similarity-grouped:** the gap is
  duplication. The paper's interpretation stands.
- **random-grouped drops substantially toward similarity-grouped:** much of the
  gap is the blocking itself, and the paper's central claim does not follow.
- **random-grouped lands between the two:** both mechanisms contribute, and the
  paper must report the decomposition rather than attribute the whole gap.

We commit to reporting whichever occurs.

## 1. Environment and data

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import tensorflow as tf, keras
print("tensorflow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
import os, json, time, shutil, random, gc
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)

RESULTS_DIR = f"{WORK}/fyp_phase2r_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED, "n_folds": 5}
N_FOLDS = 5

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("outputs ->", RESULTS_DIR)

In [ ]:
# Loader copied from Phase 2X rather than rewritten. split_seed42.csv carries
# the columns `label`, `file`, `y`, `group` -- there is NO `path` column, and
# assuming one is what made the first attempt at this notebook fail on its
# fourth cell.
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}

DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
        print("using the attached dataset:", DATA_ROOT)

if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
    DATA_ROOT = os.path.dirname(cands[0])
    print("downloaded via kagglehub:", DATA_ROOT)

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{SCRATCH}/split_seed42.csv", SPLIT_URL],
               check=True)
df = pd.read_csv(f"{SCRATCH}/split_seed42.csv")
print("columns:", list(df.columns))
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"]), "image paths do not resolve"

y_all = df["y"].to_numpy()
groups = df["group"].to_numpy()
print("images:", len(df), "| similarity groups:", df["group"].nunique())
RESULTS["data"] = {"n_images": int(len(df)), "n_groups": int(df["group"].nunique())}
save_json()

## 2. Build the random size-matched groups

The size distribution is copied exactly from the real grouping, so the two
blocked protocols differ only in *which* images share a block, never in how the
blocks are shaped.

In [ ]:
sizes = np.bincount(groups)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(df))
rand_groups = np.empty(len(df), dtype=int)
i = 0
for gid, sz in enumerate(sizes):
    rand_groups[perm[i:i + sz]] = gid
    i += sz
assert i == len(df)
assert sorted(np.bincount(rand_groups)) == sorted(sizes), \
    "random groups must match the real size distribution exactly"
print(f"size distribution matched: {len(sizes)} groups, "
      f"mean {sizes.mean():.1f}, max {sizes.max()}")

# Sanity: real groups should be far more visually coherent than random ones.
# We do not need the thumbnails again; the point is simply that the two
# groupings share their shape and differ in their content.
RESULTS["random_groups"] = {"n_groups": int(len(sizes)),
                            "mean_size": float(sizes.mean()),
                            "max_size": int(sizes.max())}
save_json()

## 3. Three protocols

The two blocked protocols use the **same splitter**, `GroupKFold`, so the only
difference between them is whether the blocks are similarity groups or random
groups. The image-level arm is the stratified convention from the literature.

In [ ]:
PROTOCOLS = {
    "image-level": list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                        random_state=SEED).split(df, y_all)),
    "random-grouped": list(GroupKFold(n_splits=N_FOLDS).split(df, y_all,
                                                             groups=rand_groups)),
    "similarity-grouped": list(GroupKFold(n_splits=N_FOLDS).split(df, y_all,
                                                                 groups=groups)),
}

check = {}
for name, folds in PROTOCOLS.items():
    real_span = [int(len(set(groups[tr]) & set(groups[te]))) for tr, te in folds]
    rand_span = [int(len(set(rand_groups[tr]) & set(rand_groups[te])))
                 for tr, te in folds]
    check[name] = {"similarity_groups_spanning": real_span,
                   "random_groups_spanning": rand_span}
    print(f"{name:20s} similarity groups spanning train/test: {real_span}")
RESULTS["protocol_check"] = check
save_json()
print()
print("Expected: similarity-grouped shows zeros for similarity groups;")
print("random-grouped does NOT, because its blocks ignore visual similarity.")

## 4. Models and the run

In [ ]:
CACHE = {}
def images_at(size):
    if size not in CACHE:
        X = np.empty((len(df), size, size, 3), np.float32)
        for i, p in enumerate(tqdm(df["path"], desc=f"load {size}px", leave=False)):
            X[i] = np.asarray(Image.open(p).convert("RGB").resize((size, size),
                                                                  Image.BILINEAR),
                              np.float32)
        CACHE[size] = X
    return CACHE[size]

def _transfer(base_fn, preprocess, size, n_classes=3, unfreeze=20):
    inp = layers.Input((size, size, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = base_fn(include_top=False, weights="imagenet",
                   input_shape=(size, size, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze]:
        layer.trainable = False
    x = base(preprocess(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(x))

ARCHS = {
    "EfficientNetB0 (pretrained)": lambda: _transfer(
        keras.applications.EfficientNetB0,
        keras.applications.efficientnet.preprocess_input, 224),
    "ResNet50 (pretrained)": lambda: _transfer(
        keras.applications.ResNet50,
        keras.applications.resnet50.preprocess_input, 224),
}
EPOCHS, BATCH, LR, SIZE = 30, 16, 1e-4, 224

def evaluate(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1.mean())}

In [ ]:
X = images_at(SIZE)
rows, t_start = [], time.time()
for arch_name, build in ARCHS.items():
    for proto_name, folds in PROTOCOLS.items():
        for k, (tr, te) in enumerate(folds):
            keras.backend.clear_session()
            tf.random.set_seed(SEED + k)
            counts = np.bincount(y_all[tr], minlength=3)
            cw = {i: float(len(tr) / (3 * c)) if c else 0.0
                  for i, c in enumerate(counts)}
            model = build()
            model.compile(optimizer=keras.optimizers.Adam(LR),
                          loss="sparse_categorical_crossentropy",
                          metrics=["accuracy"])
            t0 = time.time()
            model.fit(X[tr], y_all[tr], epochs=EPOCHS, batch_size=BATCH,
                      class_weight=cw, verbose=0)
            m = evaluate(y_all[te], model.predict(X[te], verbose=0).argmax(1))
            m.update({"architecture": arch_name, "protocol": proto_name,
                      "fold": k, "n_train": int(len(tr)), "n_test": int(len(te)),
                      "minutes": (time.time() - t0) / 60})
            rows.append(m)
            print(f"{arch_name[:14]:14s} | {proto_name:19s} | fold {k} | "
                  f"acc {m['accuracy']:.3f}  bal {m['balanced_accuracy']:.3f}  "
                  f"({m['minutes']:.1f}m)")
            del model; gc.collect()

cv = pd.DataFrame(rows)
cv.to_csv(f"{RESULTS_DIR}/cv_per_fold.csv", index=False)
RESULTS["per_fold"] = cv.to_dict("records")
print(f"\ntotal {(time.time()-t_start)/60:.1f} min over {len(cv)} runs")
save_json()

## 5. The answer

In [ ]:
summary = (cv.groupby(["architecture", "protocol"])
             .agg(acc=("accuracy", "mean"), acc_sd=("accuracy", "std"),
                  bal=("balanced_accuracy", "mean"), bal_sd=("balanced_accuracy", "std"),
                  f1=("macro_f1", "mean"))
             .round(4).reset_index())
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS_DIR}/summary.csv", index=False)
RESULTS["summary"] = summary.to_dict("records")

print()
verdict = {}
for arch in cv["architecture"].unique():
    g = lambda p: cv[(cv.architecture == arch) &
                     (cv.protocol == p)]["balanced_accuracy"].mean()
    img, rnd, sim = g("image-level"), g("random-grouped"), g("similarity-grouped")
    total = img - sim
    blocking = img - rnd          # cost of blocking alone
    duplication = rnd - sim       # cost attributable to near-duplicates
    share = duplication / total if total else float("nan")
    verdict[arch] = {"image_level": img, "random_grouped": rnd,
                     "similarity_grouped": sim, "total_gap": total,
                     "blocking_component": blocking,
                     "duplication_component": duplication,
                     "duplication_share": share}
    print(f"{arch}")
    print(f"  image-level        {img:.4f}")
    print(f"  random-grouped     {rnd:.4f}   (blocking costs {blocking:+.4f})")
    print(f"  similarity-grouped {sim:.4f}   (duplication costs {duplication:+.4f})")
    print(f"  -> {share:.1%} of the total {total:.4f} gap is attributable to")
    print(f"     near-duplicate separation rather than to blocking")
    print()
RESULTS["decomposition"] = verdict
save_json()

In [ ]:
fig, axes = plt.subplots(1, len(ARCHS), figsize=(5.5 * len(ARCHS), 4), squeeze=False)
order = ["image-level", "random-grouped", "similarity-grouped"]
colours = ["#c0392b", "#b7950b", "#2471a3"]
for ax, arch in zip(axes[0], cv["architecture"].unique()):
    for i, (proto, col) in enumerate(zip(order, colours)):
        v = cv[(cv.architecture == arch) &
               (cv.protocol == proto)]["balanced_accuracy"]
        ax.scatter([i] * len(v), v, color=col, alpha=0.3, s=22, linewidths=0)
        ax.errorbar(i, v.mean(), yerr=v.std(), fmt="o", color=col,
                    markersize=9, capsize=5, linewidth=1.8)
    ax.set_xticks(range(3))
    ax.set_xticklabels([o.replace("-", "-\n") for o in order], fontsize=9)
    ax.set_ylim(0, 1.02); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
    ax.axhline(1/3, color="grey", ls=":", lw=1)
    ax.set_title(arch, fontsize=10)
axes[0][0].set_ylabel("balanced accuracy")
fig.suptitle("Random size-matched groups isolate blocking from duplication",
             fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(f"{RESULTS_DIR}/fig_random_control.png", dpi=200)
plt.show()

In [ ]:
path = shutil.make_archive(f"{WORK}/fyp_phase2r_results", "zip", RESULTS_DIR)
print("archive:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
if not IN_KAGGLE:
    try:
        from google.colab import files; files.download(path)
    except Exception as e:
        print("download from the file browser:", e)